# Step 4: Data Preprocessing and Missing Values Handling
Here we will load the historical data from ENTSOE, preprocess and clean, and finally create a new CSV ready to be used.

### Rationale for Missing Values Imputation
Data inspection often reveals missing values caused by Daylight Saving Time (DST) transitions (e.g., the missing hour in March) or API anomalies. To address this, the time-series is explicitly resampled to a continuous hourly frequency to expose all temporal gaps. These missing values are imputed using a **forward-fill (ffill)** methodology. In short-term electricity markets, the market clearing price of the immediately preceding hour serves as the most statistically sound baseline for an anomalous missing hour, avoiding the introduction of future bias.

In [4]:
import os
from dotenv import load_dotenv
import pandas as pd
from entsoe import EntsoePandasClient
import time

# Carga las variables secretas del archivo .env local
load_dotenv()

# Lee la clave de forma segura desde el archivo oculto
api_key = os.getenv("ENTSOE_API_KEY")
client = EntsoePandasClient(api_key=api_key)

country_code = 'ES'
start = pd.Timestamp('2023-01-01', tz='Europe/Madrid')
end = pd.Timestamp('2025-12-31 23:59', tz='Europe/Madrid')

## ENTSOE-T
Next cells will be using the API token from ENTSOE for getting the data.It will be divided in 5 parts, each part with two cells (raw extraction + preprocessing). Each part corresponds to each data series we will be using, and they will be:
-Load forecast
-Day ahead market price
-Nuclear generation
-Hydro reservoir levels (MWh)
-Wind and solar forecast (STILL NEEDS TO BE DONE)


Every data series was reviewed and preprocessed accordingly to the problems it presented in raw version. Btw, everything is explained in detail in the Overleaf I sent to the group.

I kept the docs in two different folders so you can use the, because I am not sure if you will be able to use my ENTSOE-T token.

In [5]:
import os
import pandas as pd
from entsoe import EntsoePandasClient

client = EntsoePandasClient(api_key=os.getenv("ENTSOE_API_KEY"))

os.makedirs("raw", exist_ok=True)

print(" 2023...")
df_2023 = client.query_load_forecast('ES', start=pd.Timestamp('2023-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2023-12-31 23:59', tz='Europe/Madrid'))

time.sleep(2)

print(" 2024...")
df_2024 = client.query_load_forecast('ES', start=pd.Timestamp('2024-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2024-12-31 23:59', tz='Europe/Madrid'))

time.sleep(2)

print(" 2025...")
df_2025 = client.query_load_forecast('ES', start=pd.Timestamp('2025-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2025-12-31 23:59', tz='Europe/Madrid'))

df_raw = pd.concat([df_2023, df_2024, df_2025])
df_raw.to_csv("raw/Raw_LoadForecast_2023_2025_ES.csv")

print("Saved in raw/Raw_LoadForecast_2023_2025_ES.csv")

 2023...
 2024...
 2025...
Saved in raw/Raw_LoadForecast_2023_2025_ES.csv


In [6]:
import os
import pandas as pd

# 1. Ensure the directory for processed data exists
os.makedirs("processed", exist_ok=True)

print("1. Loading raw load forecast data...")
# Read the CSV assuming the date is the index (column 0)
df = pd.read_csv("raw/Raw_LoadForecast_2023_2025_ES.csv", index_col=0)

print("2. Flattening the Timezone to strict UTC...")
# Forcing utc=True converts +01:00 and +02:00 offsets to a uniform timeline.
# tz_localize(None) removes the timezone tag, leaving a purely mathematical timeline.
df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)

print("3. Resampling to 1-hour frequency and handling Autumn DST...")
# resample('1h').mean() groups every 4 quarters into 1 hour, 
# and simultaneously averages the duplicated October hour into a single row.
df_clean = df.resample('1h').mean()

print("4. Forcing exact 3-year timeline to remove UTC shifted rows from 2022...")
# Generate the perfect mathematical timeline of 26304 hours
target_index = pd.date_range(start='2023-01-01 00:00:00', end='2025-12-31 23:00:00', freq='1h')
# reindex applies the guillotine: drops 2022 rows and exposes Spring DST gaps
df_clean = df_clean.reindex(target_index)

print("5. Null values audit (Spring DST and API gaps)...")
nulls_before = df_clean.isna().sum().iloc[0]
print(f"   -> Found {nulls_before} gaps (NaN) before interpolation.")

print("6. Applying linear interpolation and final forward-fill...")
# Interpolate fills the Spring DST gaps
df_clean = df_clean.interpolate(method='linear')
# Forward-fill covers any NaN at the very last hour of 2025 due to UTC shifting
df_clean = df_clean.ffill()

nulls_after = df_clean.isna().sum().iloc[0]
print(f"   -> Remaining nulls after processing: {nulls_after}")

# Format column and index names to ensure consistency in the final model
df_clean.index.name = 'Datetime'
df_clean.columns = ['Load_Forecast_MW']

print("7. Saving processed file...")
output_path = "processed/Processed_LoadForecast_2023_2025_ES.csv"
df_clean.to_csv(output_path)

print("\n✅ CLEANING SUCCESSFULLY COMPLETED!")
print(f"Final total rows: {len(df_clean)} (The mathematical target for 3 years is exactly 26304)")
print(f"File ready and saved at: '{output_path}'")

1. Loading raw load forecast data...
2. Flattening the Timezone to strict UTC...
3. Resampling to 1-hour frequency and handling Autumn DST...
4. Forcing exact 3-year timeline to remove UTC shifted rows from 2022...
5. Null values audit (Spring DST and API gaps)...
   -> Found 1 gaps (NaN) before interpolation.
6. Applying linear interpolation and final forward-fill...
   -> Remaining nulls after processing: 0
7. Saving processed file...

✅ CLEANING SUCCESSFULLY COMPLETED!
Final total rows: 26304 (The mathematical target for 3 years is exactly 26304)
File ready and saved at: 'processed/Processed_LoadForecast_2023_2025_ES.csv'


In [7]:
import os
import pandas as pd
from entsoe import EntsoePandasClient

client = EntsoePandasClient(api_key=os.getenv("ENTSOE_API_KEY"))

os.makedirs("raw", exist_ok=True)

print("Downloading Hydro Reservoir 2023...")
df_2023 = client.query_aggregate_water_reservoirs_and_hydro_storage('ES', start=pd.Timestamp('2023-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2023-12-31 23:59', tz='Europe/Madrid'))

time.sleep(2)


print("Downloading Hydro Reservoir 2024...")
df_2024 = client.query_aggregate_water_reservoirs_and_hydro_storage('ES', start=pd.Timestamp('2024-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2024-12-31 23:59', tz='Europe/Madrid'))

time.sleep(2)


print("Downloading Hydro Reservoir 2025...")
df_2025 = client.query_aggregate_water_reservoirs_and_hydro_storage('ES', start=pd.Timestamp('2025-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2025-12-31 23:59', tz='Europe/Madrid'))

# Merge the three years
df_raw = pd.concat([df_2023, df_2024, df_2025])
df_raw.to_csv("raw/Raw_HydroReservoir_2023_2025_ES.csv")

print("Download completed and saved in raw/Raw_HydroReservoir_2023_2025_ES.csv")

Download completed and saved in raw/Raw_HydroReservoir_2023_2025_ES.csv


In [8]:
import os
import pandas as pd



# 1. Ensure the directory for processed data exists
os.makedirs("processed", exist_ok=True)

print("1. Loading raw hydro reservoir data...")
df = pd.read_csv("raw/Raw_HydroReservoir_2023_2025_ES.csv", index_col=0)

print("2. Renaming column and formatting index to strict UTC...")
df.columns = ['Hydro_Reservoir_MWh']
df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
df.index.name = 'Datetime'

print("3. Removing duplicate overlapping weeks from API concatenation...")
df = df[~df.index.duplicated(keep='first')]

print("4. Upsampling to hourly resolution and applying forward-fill...")
df_hourly = df.resample('1h').ffill()

print("5. Forcing exact 3-year timeline (padding missing final days)...")
# Generate the perfect mathematical timeline of 26304 hours
target_index = pd.date_range(start='2023-01-01 00:00:00', end='2025-12-31 23:00:00', freq='1h')

# reindex adapts the dataframe to our target_index, trimming 2022 and stretching the end of 2025
df_clean = df_hourly.reindex(target_index).ffill()
df_clean.index.name = 'Datetime'

print("6. Saving processed file...")
output_path = "processed/Processed_HydroReservoir_2023_2025_ES.csv"
df_clean.to_csv(output_path)

print("\n✅ CLEANING SUCCESSFULLY COMPLETED!")
print(f"Final total rows: {len(df_clean)} (The mathematical target is exactly 26304)")
print(f"File ready and saved at: '{output_path}'")

1. Loading raw hydro reservoir data...
2. Renaming column and formatting index to strict UTC...
3. Removing duplicate overlapping weeks from API concatenation...
4. Upsampling to hourly resolution and applying forward-fill...
5. Forcing exact 3-year timeline (padding missing final days)...
6. Saving processed file...

✅ CLEANING SUCCESSFULLY COMPLETED!
Final total rows: 26304 (The mathematical target is exactly 26304)
File ready and saved at: 'processed/Processed_HydroReservoir_2023_2025_ES.csv'


In [9]:
import os
import pandas as pd
from entsoe import EntsoePandasClient

client = EntsoePandasClient(api_key=os.getenv("ENTSOE_API_KEY"))

os.makedirs("raw", exist_ok=True)

print("Downloading Day-ahead Prices 2023...")
df_2023 = client.query_day_ahead_prices('ES', start=pd.Timestamp('2023-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2023-12-31 23:59', tz='Europe/Madrid'))

time.sleep(2)


print("Downloading Day-ahead Prices 2024...")
df_2024 = client.query_day_ahead_prices('ES', start=pd.Timestamp('2024-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2024-12-31 23:59', tz='Europe/Madrid'))

time.sleep(2)


print("Downloading Day-ahead Prices 2025...")
df_2025 = client.query_day_ahead_prices('ES', start=pd.Timestamp('2025-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2025-12-31 23:59', tz='Europe/Madrid'))

# Merge the three years
df_raw = pd.concat([df_2023, df_2024, df_2025])
df_raw.to_csv("raw/Raw_DayAheadPrices_2023_2025_ES.csv")

print("Download completed and saved in raw/Raw_DayAheadPrices_2023_2025_ES.csv")

Download completed and saved in raw/Raw_DayAheadPrices_2023_2025_ES.csv


In [10]:
import os
import pandas as pd

# 1. Ensure the directory for processed data exists
os.makedirs("processed", exist_ok=True)

print("1. Loading raw day-ahead prices data...")
df = pd.read_csv("raw/Raw_DayAheadPrices_2023_2025_ES.csv", index_col=0)

print("2. Renaming column and formatting index to strict UTC...")
# ENTSO-E usually names the column '0', we rename it for clarity
df.columns = ['Price_EUR_MWh']
df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
df.index.name = 'Datetime'

print("3. Homogenizing resolution (1h) and handling Autumn DST...")
# resample('1h').mean() does three things at once:
# - Leaves 2023-2024 hourly data intact
# - Averages the 15-min blocks from Oct 2025 onwards into hourly values
# - Merges the duplicated hour during Autumn DST into a single average value
df_hourly = df.resample('1h').mean()

print("4. Forcing exact 3-year timeline to expose Spring DST gaps...")
# Generate the perfect mathematical timeline of 26304 hours
target_index = pd.date_range(start='2023-01-01 00:00:00', end='2025-12-31 23:00:00', freq='1h')
df_clean = df_hourly.reindex(target_index)
df_clean.index.name = 'Datetime'

print("5. Null values audit and linear interpolation...")
nulls_before = df_clean.isna().sum().iloc[0]
print(f"   -> Found {nulls_before} missing hours (NaN) before interpolation.")

# Interpolate to fill Spring DST missing hours mathematically
df_clean = df_clean.interpolate(method='linear')
nulls_after = df_clean.isna().sum().iloc[0]
print(f"   -> Remaining nulls after interpolation: {nulls_after}")

print("6. Saving processed file...")
output_path = "processed/Processed_DayAheadPrices_2023_2025_ES.csv"
df_clean.to_csv(output_path)

print("\n✅ CLEANING SUCCESSFULLY COMPLETED!")
print(f"Final total rows: {len(df_clean)} (The mathematical target is exactly 26304)")
print(f"File ready and saved at: '{output_path}'")

1. Loading raw day-ahead prices data...
2. Renaming column and formatting index to strict UTC...
3. Homogenizing resolution (1h) and handling Autumn DST...
4. Forcing exact 3-year timeline to expose Spring DST gaps...
5. Null values audit and linear interpolation...
   -> Found 3 missing hours (NaN) before interpolation.
   -> Remaining nulls after interpolation: 0
6. Saving processed file...

✅ CLEANING SUCCESSFULLY COMPLETED!
Final total rows: 26304 (The mathematical target is exactly 26304)
File ready and saved at: 'processed/Processed_DayAheadPrices_2023_2025_ES.csv'


In [11]:
import os
import pandas as pd
from entsoe import EntsoePandasClient

client = EntsoePandasClient(api_key=os.getenv("ENTSOE_API_KEY"))

os.makedirs("raw", exist_ok=True)

print("Downloading Actual Generation 2023 (this might take a few minutes)...")
df_2023 = client.query_generation('ES', start=pd.Timestamp('2023-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2023-12-31 23:59', tz='Europe/Madrid'))

time.sleep(2)


print("Downloading Actual Generation 2024...")
df_2024 = client.query_generation('ES', start=pd.Timestamp('2024-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2024-12-31 23:59', tz='Europe/Madrid'))

time.sleep(2)


print("Downloading Actual Generation 2025...")
df_2025 = client.query_generation('ES', start=pd.Timestamp('2025-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2025-12-31 23:59', tz='Europe/Madrid'))

time.sleep(2)


# Merge the three years
df_raw = pd.concat([df_2023, df_2024, df_2025])

# Flatten columns if ENTSO-E returns a MultiIndex (common in query_generation)
if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = df_raw.columns.droplevel(1)

# Extract ONLY the Nuclear column to keep the raw file clean
nuclear_cols = [col for col in df_raw.columns if 'Nuclear' in str(col)]
df_nuclear = df_raw[nuclear_cols].copy()

# Rename for clarity
df_nuclear.columns = ['Nuclear_Generation_MW']

output_filename = "raw/Raw_NuclearGeneration_2023_2025_ES.csv"
df_nuclear.to_csv(output_filename)

print(f"Download completed and saved in {output_filename}")

Download completed and saved in raw/Raw_NuclearGeneration_2023_2025_ES.csv


In [12]:
import os
import pandas as pd

# 1. Ensure the directory for processed data exists
os.makedirs("processed", exist_ok=True)

print("1. Loading raw nuclear generation data...")
df = pd.read_csv("raw/Raw_NuclearGeneration_2023_2025_ES.csv", index_col=0)



print("2. Formatting index to strict UTC...")
df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
df.index.name = 'Datetime'

print("3. Homogenizing resolution (1h) and handling Autumn DST...")
# resample('1h').mean() compresses the 15-min blocks into hourly values
# and automatically merges the duplicated hour during Autumn DST
df_hourly = df.resample('1h').mean()

print("4. Forcing exact 3-year timeline to expose telemetry gaps and Spring DST...")
# Generate the perfect mathematical timeline of 26304 hours
target_index = pd.date_range(start='2023-01-01 00:00:00', end='2025-12-31 23:00:00', freq='1h')
df_clean = df_hourly.reindex(target_index)
df_clean.index.name = 'Datetime'

print("5. Null values audit and linear interpolation...")
nulls_before = df_clean.isna().sum().iloc[0]
print(f"   -> Found {nulls_before} missing hours (NaN) before interpolation.")

# Interpolate to mathematically fill telemetry micro-cuts and Spring DST
df_clean = df_clean.interpolate(method='linear')
nulls_after = df_clean.isna().sum().iloc[0]
print(f"   -> Remaining nulls after interpolation: {nulls_after}")

print("6. Saving processed file...")
output_path = "processed/Processed_NuclearGeneration_2023_2025_ES.csv"
df_clean.to_csv(output_path)

print("\n✅ CLEANING SUCCESSFULLY COMPLETED!")
print(f"Final total rows: {len(df_clean)} (The mathematical target is exactly 26304)")
print(f"File ready and saved at: '{output_path}'")

1. Loading raw nuclear generation data...
2. Formatting index to strict UTC...
3. Homogenizing resolution (1h) and handling Autumn DST...
4. Forcing exact 3-year timeline to expose telemetry gaps and Spring DST...
5. Null values audit and linear interpolation...
   -> Found 36 missing hours (NaN) before interpolation.
   -> Remaining nulls after interpolation: 0
6. Saving processed file...

✅ CLEANING SUCCESSFULLY COMPLETED!
Final total rows: 26304 (The mathematical target is exactly 26304)
File ready and saved at: 'processed/Processed_NuclearGeneration_2023_2025_ES.csv'


In [13]:
import os
import pandas as pd
from entsoe import EntsoePandasClient

client = EntsoePandasClient(api_key=os.getenv("ENTSOE_API_KEY"))

os.makedirs("raw", exist_ok=True)

print("Downloading Wind and Solar Forecast 2023...")
df_2023 = client.query_wind_and_solar_forecast('ES', start=pd.Timestamp('2023-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2023-12-31 23:59', tz='Europe/Madrid'))

time.sleep(2)


print("Downloading Wind and Solar Forecast 2024...")
df_2024 = client.query_wind_and_solar_forecast('ES', start=pd.Timestamp('2024-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2024-12-31 23:59', tz='Europe/Madrid'))

time.sleep(2)


print("Downloading Wind and Solar Forecast 2025...")
df_2025 = client.query_wind_and_solar_forecast('ES', start=pd.Timestamp('2025-01-01', tz='Europe/Madrid'), end=pd.Timestamp('2025-12-31 23:59', tz='Europe/Madrid'))

# Merge the three years
df_raw = pd.concat([df_2023, df_2024, df_2025])

output_filename = "raw/Raw_WindSolarForecast_2023_2025_ES.csv"
df_raw.to_csv(output_filename)

print(f"Download completed and saved in {output_filename}")

Download completed and saved in raw/Raw_WindSolarForecast_2023_2025_ES.csv


In [14]:
import os
import pandas as pd

# 1. Ensure the directory for processed data exists
os.makedirs("processed", exist_ok=True)

print("1. Loading raw wind and solar forecast data...")
df = pd.read_csv("raw/Raw_WindSolarForecast_2023_2025_ES.csv", index_col=0)

print("2. Filtering target columns...")
# Identify and keep only the relevant columns for Spain (ignoring Offshore Wind)
wind_col = [col for col in df.columns if 'Wind Onshore' in col][0]
solar_col = [col for col in df.columns if 'Solar' in col][0]

df = df[[wind_col, solar_col]].copy()

print("3. Formatting index to strict UTC...")
df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
df.index.name = 'Datetime'

print("4. Homogenizing resolution (1h) and handling Autumn DST...")
# Compresses the 15-min blocks and merges the duplicated Autumn DST hour
df_hourly = df.resample('1h').mean()

print("5. Forcing exact 3-year timeline to expose telemetry gaps and Spring DST...")
# Generate the perfect mathematical timeline of 26304 hours
target_index = pd.date_range(start='2023-01-01 00:00:00', end='2025-12-31 23:00:00', freq='1h')
df_clean = df_hourly.reindex(target_index)
df_clean.index.name = 'Datetime'

print("6. Null values audit and linear interpolation...")
nulls_before = df_clean.isna().sum()
print("   -> Missing hours (NaN) before interpolation:\n", nulls_before)

# Interpolate to mathematically fill telemetry micro-cuts and Spring DST
df_clean = df_clean.interpolate(method='linear')
nulls_after = df_clean.isna().sum()
print("   -> Remaining nulls after interpolation:\n", nulls_after)

# Final renaming for the master dataset
df_clean.columns = ['Wind_Onshore_MW', 'Solar_MW']

print("7. Saving processed file...")
output_path = "processed/Processed_WindSolarForecast_2023_2025_ES.csv"
df_clean.to_csv(output_path)

print("\n✅ CLEANING SUCCESSFULLY COMPLETED!")
print(f"Final total rows: {len(df_clean)} (The mathematical target is exactly 26304)")
print(f"File ready and saved at: '{output_path}'")

1. Loading raw wind and solar forecast data...
2. Filtering target columns...
3. Formatting index to strict UTC...
4. Homogenizing resolution (1h) and handling Autumn DST...
5. Forcing exact 3-year timeline to expose telemetry gaps and Spring DST...
6. Null values audit and linear interpolation...
   -> Missing hours (NaN) before interpolation:
 Wind Onshore     49
Solar           121
dtype: int64
   -> Remaining nulls after interpolation:
 Wind Onshore    0
Solar           0
dtype: int64
7. Saving processed file...

✅ CLEANING SUCCESSFULLY COMPLETED!
Final total rows: 26304 (The mathematical target is exactly 26304)
File ready and saved at: 'processed/Processed_WindSolarForecast_2023_2025_ES.csv'


In [15]:
import os
import pandas as pd

print("1. Loading all processed datasets...")
# Using parse_dates=True ensures Pandas treats the 'Datetime' index correctly for merging
df_prices = pd.read_csv("processed/Processed_DayAheadPrices_2023_2025_ES.csv", index_col='Datetime', parse_dates=True)
df_load = pd.read_csv("processed/Processed_LoadForecast_2023_2025_ES.csv", index_col='Datetime', parse_dates=True)
df_hydro = pd.read_csv("processed/Processed_HydroReservoir_2023_2025_ES.csv", index_col='Datetime', parse_dates=True)
df_nuclear = pd.read_csv("processed/Processed_NuclearGeneration_2023_2025_ES.csv", index_col='Datetime', parse_dates=True)
df_wind_solar = pd.read_csv("processed/Processed_WindSolarForecast_2023_2025_ES.csv", index_col='Datetime', parse_dates=True)

print("2. Merging datasets into a single Master DataFrame...")
# pd.concat horizontally (axis=1) merges all columns matching the perfect 26304-hour index
df_master = pd.concat([df_prices, df_load, df_hydro, df_nuclear, df_wind_solar], axis=1)

print("3. Saving the base Master Dataset...")
# We save this base version in 'processed' before adding lagged/calendar features
os.makedirs("processed", exist_ok=True)
output_path = "processed/Master_Dataset_2023_2025.csv"
df_master.to_csv(output_path)

print("\n✅ BASE MASTER DATASET SUCCESSFULLY CREATED!")
print(f"Total rows: {len(df_master)}")
print(f"Included features: {list(df_master.columns)}")
print(f"File saved at: '{output_path}'")

1. Loading all processed datasets...
2. Merging datasets into a single Master DataFrame...
3. Saving the base Master Dataset...

✅ BASE MASTER DATASET SUCCESSFULLY CREATED!
Total rows: 26304
Included features: ['Price_EUR_MWh', 'Load_Forecast_MW', 'Hydro_Reservoir_MWh', 'Nuclear_Generation_MW', 'Wind_Onshore_MW', 'Solar_MW']
File saved at: 'processed/Master_Dataset_2023_2025.csv'


In [16]:
import os
import pandas as pd

print("1. Loading the base Master Dataset...")
# Read the merged dataset, ensuring the index is treated as a datetime object
df_master = pd.read_csv("processed/Master_Dataset_2023_2025.csv", index_col='Datetime', parse_dates=True)

print("2. Creating Lagged Features for Day-ahead Prices...")
# Deriving historical price lags (1 day, 2 days, and 1 week ago)
df_master['Price_t_24'] = df_master['Price_EUR_MWh'].shift(24)
df_master['Price_t_48'] = df_master['Price_EUR_MWh'].shift(48)
df_master['Price_t_168'] = df_master['Price_EUR_MWh'].shift(168)

# 24-hour moving average. Note: shift(1) is CRITICAL to avoid data leakage, 
# ensuring we don't include the current hour's price in its own prediction feature.
df_master['Price_24h_MA'] = df_master['Price_EUR_MWh'].shift(24).rolling(window=24).mean()

print("3. Creating Calendar Features...")
# Extracting temporal patterns for the neural network
df_master['Hour'] = df_master.index.hour
df_master['Month'] = df_master.index.month

# dayofweek: 0=Monday to 6=Sunday. Values >= 5 correspond to the weekend.
df_master['Is_Weekend'] = (df_master.index.dayofweek >= 5).astype(int) 

print("4. Dropping rows with NaNs generated by time lags...")
# The shift(168) creates 168 empty rows at the beginning of 2023 because we don't have 2022 data.
df_master_clean = df_master.dropna()

print("5. Saving the final Engineered Dataset...")
# We save this final version in a new folder called 'model_input'
os.makedirs("model_input", exist_ok=True)
output_path = "model_input/Master_Dataset_Engineered_2023_2025.csv"
df_master_clean.to_csv(output_path)

print("\n✅ FEATURE ENGINEERING SUCCESSFULLY COMPLETED!")
print(f"Final rows (after dropping initial 168h lag): {len(df_master_clean)}")
print(f"Total features (columns): {len(df_master_clean.columns)}")
print(f"File ready and saved at: '{output_path}'")

1. Loading the base Master Dataset...
2. Creating Lagged Features for Day-ahead Prices...
3. Creating Calendar Features...
4. Dropping rows with NaNs generated by time lags...
5. Saving the final Engineered Dataset...

✅ FEATURE ENGINEERING SUCCESSFULLY COMPLETED!
Final rows (after dropping initial 168h lag): 26136
Total features (columns): 13
File ready and saved at: 'model_input/Master_Dataset_Engineered_2023_2025.csv'


In [18]:
import pandas as pd
import os

print("1. Loading Master Dataset from CSV...")
csv_path = "model_input/Master_Dataset_Engineered_2023_2025.csv"
df = pd.read_csv(csv_path)

print("2. Converting to Excel format...")
excel_path = "model_input/Master_Dataset_Engineered_2023_2025.xlsx"
df.to_excel(excel_path, index=False)

print("\n✅ EXCEL FILE SUCCESSFULLY GENERATED")
print(f"The file is saved at: '{excel_path}'")

1. Loading Master Dataset from CSV...
2. Converting to Excel format...

✅ EXCEL FILE SUCCESSFULLY GENERATED
The file is saved at: 'model_input/Master_Dataset_Engineered_2023_2025.xlsx'
